In [ ]:
!pip install ragas langchain langchain-community langchain-google-genai datasets

In [ ]:
!pip install langchain-google-vertexai

In [ ]:
import sys
import types

# --- QUICK FIX FOR RAGAS BUG ---
# Ragas is crashing because it's looking for a deleted VertexAI module.
# We are creating a dummy module so it loads successfully.
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import json
import os
import time
from ragas import aevaluate
from ragas.metrics import faithfulness, answer_relevancy
from datasets import Dataset
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

from google.colab import userdata

# Safely retrieve the key from the Secrets manager
google_api_key = userdata.get('GOOGLE_API_KEY')

# Use it in your model initialization
gemini_model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=google_api_key
)


evaluator_llm = LangchainLLMWrapper(gemini_model)

# 3. Embeddings
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# 4. Load Data
with open("eval_dataset.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_dict({
    "question": data["question"],
    "contexts": data["retrieved_contexts"],
    "answer": data["answer"],
    "ground_truth": data["ground_truth"],
})

# 5. The "Anti-Rate-Limit" Wrapper
# We process metrics one by one to avoid overwhelming the API
async def safe_evaluate(dataset, metric):
    print(f"Evaluating metric: {metric.name}...")
    # Throttling: RunConfig ensures only 1 worker,
    # and we add a small wait to be extra safe
    results = await aevaluate(
        dataset,
        metrics=[metric],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=RunConfig(max_workers=1, timeout=600)
    )
    time.sleep(5)  # Wait 5 seconds after each metric batch
    return results

# 6. Run Evaluation
async def main():
    res = await safe_evaluate(dataset, faithfulness)

    print(f"Finished faithfulness")

    print(res)

if __name__ == "__main__":
    await main()

/tmp/ipykernel_6571/3559843899.py:15: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_6571/3559843899.py:15: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_6571/3559843899.py:35: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gemini_model)
/tmp/ipykernel_6571/3559843899.p

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating metric: faithfulness...


/tmp/ipykernel_6571/3559843899.py:39: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)
/tmp/ipykernel_6571/3559843899.py:58: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  results = await aevaluate(


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[9]: ChatGoogleGenerativeAIError(Error calling model 'models/gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 15.782298176s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'qu

Finished faithfulness
{'faithfulness': 0.9778}


In [2]:
import sys
import types

# --- QUICK FIX FOR RAGAS BUG ---
# Ragas is crashing because it's looking for a deleted VertexAI module.
# We are creating a dummy module so it loads successfully.
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat
import asyncio
import json
import os
import time
from ragas import aevaluate
from ragas.metrics import  answer_relevancy
from datasets import Dataset
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

from google.colab import userdata

# Safely retrieve the key from the Secrets manager
google_api_key = userdata.get('GOOGLE_API_KEY2')

# Use it in your model initialization
gemini_model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=google_api_key
)


evaluator_llm = LangchainLLMWrapper(gemini_model)

# 3. Embeddings
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# 4. Load Data
with open("eval_dataset.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_dict({
    "question": data["question"],
    "contexts": data["retrieved_contexts"],
    "answer": data["answer"],
    "ground_truth": data["ground_truth"],
})

# 5. The "Anti-Rate-Limit" Wrapper
# We process metrics one by one to avoid overwhelming the API
async def safe_evaluate(dataset, metric):
    print(f"Evaluating metric: {metric.name}...")
    # Throttling: RunConfig ensures only 1 worker,
    # and we add a small wait to be extra safe
    results = await aevaluate(
        dataset,
        metrics=[metric],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=RunConfig(max_workers=1, timeout=600)
    )
    time.sleep(5)  # Wait 5 seconds after each metric batch
    return results

# 6. Run Evaluation
async def main():
    res = await safe_evaluate(dataset,  answer_relevancy)

    print(f"Finished  answer_relevancy")

    print(res)

if __name__ == "__main__":
    await main()

/tmp/ipykernel_6571/745776448.py:15: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import  answer_relevancy
/tmp/ipykernel_6571/745776448.py:35: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gemini_model)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating metric: answer_relevancy...


/tmp/ipykernel_6571/745776448.py:39: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)
/tmp/ipykernel_6571/745776448.py:58: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  results = await aevaluate(


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

Finished  answer_relevancy
{'answer_relevancy': 0.9512}


In [4]:
import sys
import types

# --- QUICK FIX FOR RAGAS BUG ---
# Ragas is crashing because it's looking for a deleted VertexAI module.
# We are creating a dummy module so it loads successfully.
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat
import asyncio
import json
import os
import time
from ragas import aevaluate
from ragas.metrics import ContextPrecision
from datasets import Dataset
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

from google.colab import userdata

# Safely retrieve the key from the Secrets manager
google_api_key = userdata.get('GOOGLE_API_KEY3')

# Use it in your model initialization
gemini_model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=google_api_key
)


evaluator_llm = LangchainLLMWrapper(gemini_model)

# 3. Embeddings
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# 4. Load Data
with open("eval_dataset.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_dict({
    "question": data["question"],
    "contexts": data["retrieved_contexts"],
    "answer": data["answer"],
    "ground_truth": data["ground_truth"],
})

# 5. The "Anti-Rate-Limit" Wrapper
# We process metrics one by one to avoid overwhelming the API
async def safe_evaluate(dataset, metric):
    print(f"Evaluating metric: { metric.name}...")
    # Throttling: RunConfig ensures only 1 worker,
    # and we add a small wait to be extra safe
    try:
      results = await aevaluate(
        dataset,
        metrics=[metric],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=RunConfig(max_workers=1, timeout=600)
      )
      time.sleep(5)  # Wait 5 seconds after each metric batch
      return results
    except Exception as e:
      print("Error in context precision: ",e)
      return None

# 6. Run Evaluation
async def main():
    metric_obj = ContextPrecision()
    res = await safe_evaluate(dataset, metric_obj)

    print(f"Finished  Context Precision")

    print(res)

if __name__ == "__main__":
    await main()

/tmp/ipykernel_17677/3943635337.py:15: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision
/tmp/ipykernel_17677/3943635337.py:35: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gemini_model)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating metric: context_precision...


/tmp/ipykernel_17677/3943635337.py:39: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)
/tmp/ipykernel_17677/3943635337.py:59: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  results = await aevaluate(


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

Finished  Context Precision
{'context_precision': 1.0000}


In [5]:
import sys
import types

# --- QUICK FIX FOR RAGAS BUG ---
# Ragas is crashing because it's looking for a deleted VertexAI module.
# We are creating a dummy module so it loads successfully.
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat
import asyncio
import json
import os
import time
from ragas import aevaluate
from ragas.metrics import ContextRecall
from datasets import Dataset
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

from google.colab import userdata

# Safely retrieve the key from the Secrets manager
google_api_key = userdata.get('GOOGLE_API_KEY3')

# Use it in your model initialization
gemini_model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=google_api_key
)


evaluator_llm = LangchainLLMWrapper(gemini_model)

# 3. Embeddings
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# 4. Load Data
with open("eval_dataset.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_dict({
    "question": data["question"],
    "contexts": data["retrieved_contexts"],
    "answer": data["answer"],
    "ground_truth": data["ground_truth"],
})

# 5. The "Anti-Rate-Limit" Wrapper
# We process metrics one by one to avoid overwhelming the API
async def safe_evaluate(dataset, metric):
    print(f"Evaluating metric: { metric.name}...")
    # Throttling: RunConfig ensures only 1 worker,
    # and we add a small wait to be extra safe
    try:
      results = await aevaluate(
        dataset,
        metrics=[metric],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=RunConfig(max_workers=1, timeout=600)
      )
      time.sleep(5)  # Wait 5 seconds after each metric batch
      return results
    except Exception as e:
      print("Error in context precision: ",e)
      return None

# 6. Run Evaluation
async def main():
    metric_obj = ContextRecall()
    res = await safe_evaluate(dataset, metric_obj)

    print(f"Finished  Context Precision")

    print(res)

if __name__ == "__main__":
    await main()

/tmp/ipykernel_17677/3302906878.py:15: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextRecall
/tmp/ipykernel_17677/3302906878.py:35: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gemini_model)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating metric: context_recall...


/tmp/ipykernel_17677/3302906878.py:39: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)
/tmp/ipykernel_17677/3302906878.py:59: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  results = await aevaluate(


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

Finished  Context Precision
{'context_recall': 1.0000}
